In [1]:
from langgraph.types import Command, interrupt
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

from dotenv import load_dotenv
load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

# 사람의 도움을 요청하는 도구 정의
@tool
def human_assistance(query:str):
    """사람의 도움이 필요할 때 호출되는 도구입니다."""
    human_response = interrupt({"query":query}) # 실행 일시 중단
    return human_response["data"]

In [2]:
# 도구 목록 추가

search_tool = TavilySearch(max_results=2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

In [3]:
# 모듈 임포트, 상태 정의, chatbot 노드 정의

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages:Annotated[list, add_messages]

def chatbot(state:State):
    message = llm_with_tools.invoke(state["messages"])
    return {"messages":[message]}

In [10]:
# 그래프 초기화 및 메모리 설정
graph_builder = StateGraph(State)
memory = MemorySaver()

# 노드 및 엣지 추가
graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node) # (노드 이름, 실행할 수 있는 것)
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START,"chatbot")

# 그래프 컴파일
graph = graph_builder.compile(checkpointer=memory)

In [12]:
# human_assistance 도구를 호출하도록 입력 전달

user_input = "네"
config = {"configurable":{"thread_id":"1"}}
events = graph.stream({"messages":[{"role":"user", "content":user_input}]}, config, stream_mode="values")
for event in events:
    if "messages" in event: event["messages"][-1].pretty_print()

================================ Human Message =================================

네
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_7LjLZUb4Z9vvTnJNteXkI7fM)
 Call ID: call_7LjLZUb4Z9vvTnJNteXkI7fM
  Args:
    query: AI 에이전트 개발 전문가의 조언을 받고 싶습니다. 전문가와 연결을 원합니다.
================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_7LjLZUb4Z9vvTnJNteXkI7fM)
 Call ID: call_7LjLZUb4Z9vvTnJNteXkI7fM
  Args:
    query: AI 에이전트 개발 전문가의 조언을 받고 싶습니다. 전문가와 연결을 원합니다.


In [13]:
snapshot = graph.get_state(config)
print(snapshot.next)

('tools',)


In [15]:
from langgraph.types import Command

human_response = (
    "전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어보세요."
    "간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다."
)
human_command = Command(resume={"data":human_response})
events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (call_7LjLZUb4Z9vvTnJNteXkI7fM)
 Call ID: call_7LjLZUb4Z9vvTnJNteXkI7fM
  Args:
    query: AI 에이전트 개발 전문가의 조언을 받고 싶습니다. 전문가와 연결을 원합니다.
================================= Tool Message =================================
Name: human_assistance

전문가입니다. LangGraph를 활용해 AI 에이전트를 만들어보세요.간단한 자율 에이전트보다 훨씬 더 확장성 있고 안정적입니다.
================================== Ai Message ==================================

전문가의 조언입니다!

AI 에이전트를 개발할 때 LangGraph라는 라이브러리를 활용해보라고 추천합니다. LangGraph는 기존의 단순한 자율 에이전트보다 훨씬 더 확장성이 크고 안정적이라고 합니다.

추가적으로 궁금한 점(예: LangGraph 설치법, 사용 예제, 어떤 AI 모델과 연동 가능한지 등)이 있거나, 다른 방향의 조언이 필요하시면 언제든 말씀해 주세요. 구체적인 상담이 필요한 경우 추가 연결도 도와드리겠습니다.
